# 03 — Basket and assistant evidence
Updated with AI assistance; docs/AI_USE.md. Read saved temporal evaluations instead
of introducing a different last-month or leave-one-item-out test.

In [ ]:
import json
import sys
from pathlib import Path
import pandas as pd
from IPython.display import display, Image
ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
def report(name):
    return json.loads((ROOT / "reports/current_run" / name).read_text())

In [ ]:
basket = report("recommendation_metrics.json")
print("Report sections:", list(basket))
display(pd.DataFrame(basket["sweep_validation"]))
print("Timing:", basket["header"]["timing"])
print("Rule keys equal:", basket["header"]["apriori_fpgrowth_rule_keys_equal"])
print("Memory:", basket["header"]["memory_report"])
for key,value in basket.items():
    if "test" in key:
        print(key, value)

## Rule interpretation and denominators
A is the visible antecedent and B is the consequent. Support counts baskets containing their full
combined itemset. Confidence is P(B|A); lift is P(B|A)/P(B). Minimum support 0.5% requires at least
129 of 25,601 training baskets. Confidence is at least 10%, lift 1.2 and combined itemset size ≤3.
No genuinely high-confidence discarded low-lift example was found; do not label a 17% example high.
Hit@5 tests whether any of five suggestions recovers a hidden item after half-basket masking.
Catalogue coverage 7.7% and rule-firing coverage 83.7% have different denominators.

In [ ]:
from src.recommend import RecommendationEngine
engine = RecommendationEngine.load()
display(engine.recommend(["white hanging heart t-light holder"], top_k=5))
print("Unmatched basket:", engine.recommend(["never-seen-item"], top_k=5)["strategy"])

## Head C: routing is not written-answer correctness
The 40 questions are development evidence, not an independent unseen evaluation. Historical
answer correctness has one AI rater. Read its limitations before using the results.
The recent low-similarity handoff is covered by explicit tests, not by retroactively changing
historical provider evaluations.

In [ ]:
rows=[]
for provider in ["regex","openai","ollama"]:
    result=report(f"assistant_{provider}_metrics.json")
    rows.append({"provider":provider,"questions":result["golden_count"],"tool_choice_accuracy":result["tool_choice_accuracy_all"],"out_of_scope_refusal":result["out_of_scope_refusal_rate"]})
display(pd.DataFrame(rows))
correctness=report("answer_correctness.json")
print("Rater:", correctness["rater"])
print("OpenAI 11/18 and Ollama 3/18 historically judged correct; independently re-evaluate new French questions.")